In [6]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import mlflow
import mlflow.xgboost
import shap

# Set plotting theme
sns.set_theme(style="whitegrid")

# Dynamically resolve root project directory mapping
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PATH = PROJECT_ROOT / "project-1-credit-card-fraud" / "data" / "processed_features.parquet"

# Point MLflow to your local tracking store directory 
mlflow.set_tracking_uri("file://" + str(PROJECT_ROOT / "mlruns"))

print("📖 Extracting latest production baseline from MLflow Model Registry...")
# Pull your trained model binary back directly from the local MLflow backend
model_uri = "models:/fraud_detection_xgb/1"
try:
    model = mlflow.xgboost.load_model(model_uri)
except mlflow.exceptions.MlflowException:
    print(f"Registered model not found at {model_uri}; falling back to local MLflow artifact search.")
    local_model_path = None
    root = PROJECT_ROOT / "mlruns"
    for exp_dir in root.iterdir():
        if not exp_dir.is_dir():
            continue
        for run_dir in exp_dir.iterdir():
            candidate = run_dir / "artifacts" / "model"
            if (candidate / "MLmodel").exists():
                local_model_path = candidate.as_posix()
                break
        if local_model_path:
            break
    if local_model_path is None:
        for exp_dir in root.iterdir():
            if not exp_dir.is_dir():
                continue
            for run_dir in exp_dir.iterdir():
                if not run_dir.is_dir():
                    continue
                candidate = run_dir / "artifacts" / "model"
                if (candidate / "MLmodel").exists():
                    local_model_path = candidate.as_posix()
                    break
            if local_model_path:
                break
        if local_model_path is None:
            raise FileNotFoundError(
                f"Local MLflow model artifact not found under {root}. "
                "Make sure the training run exists and that the model artifact is present "
                "in the local mlruns directory."
            )
    model = mlflow.xgboost.load_model(local_model_path)
    print(f"✅ Loaded local model artifact from {local_model_path}")

# Load data for evaluation footprint
df = pd.read_parquet(DATA_PATH)
X = df.drop(columns=['Class'])
y = df['Class']

print("✅ Production model binary loaded and ready for compliance audit.")

📖 Extracting latest production baseline from MLflow Model Registry...
Registered model not found at models:/fraud_detection_xgb/1; falling back to local MLflow artifact search.


FileNotFoundError: Local MLflow model artifact not found under /media/storage/mlops-governance-reference/project-1-credit-card-fraud/mlruns. Make sure the training run exists and that the model artifact is present in the local mlruns directory.

In [ ]:
print("🔮 Initializing TreeExplainer engine matrix...")
# Extract exact Tree-based Shapley value matrices for your model properties
explainer = shap.TreeExplainer(model)
shap_values = explainer(X.head(2000)) # Audit a massive 2000 transaction baseline signature

print("\n📊 Generating Global Feature Importance Audit Summary:")
# Plot global feature impact distributions to visually spot hidden proxies
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X.head(2000), show=False)
plt.title("SHAP Global Feature Importance Audit", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Locate a positive fraud example transaction from your matrix
fraud_indices = np.where(y.head(2000) == 1)[0]
if len(fraud_indices) > 0:
    target_idx = fraud_indices[0]
    print(f"🔍 Auditing High-Risk Alert Ledger Location: Index {target_idx}")
    
    # Generate local prediction force layout tracking
    plt.figure(figsize=(12, 4))
    shap.plots.waterfall(shap_values[target_idx], show=False)
    plt.title(f"Local Transaction Risk Contribution Breakdown (Index {target_idx})", fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No fraud instances located in the checked baseline sample window.")

In [ ]:
print("⚖️ Initializing Proxy Segment Disparate Impact Review...")

# Financial datasets often evaluate transaction sizes (Amounts) to check for bias against low-income users.
# Let's verify if the model penalizes low-value transactions disproportionately compared to high-value ones.
high_value_mask = X.head(2000)['scaled_amount'] > 1.0
predictions = model.predict(X.head(2000))

high_val_alert_rate = np.mean(predictions[high_value_mask] == 1)
low_val_alert_rate = np.mean(predictions[~high_value_mask] == 1)

disparate_impact_ratio = low_val_alert_rate / (high_val_alert_rate if high_val_alert_rate > 0 else 1)

print(f" - High-Value Transaction Flagging Rate: {high_val_alert_rate:.4f}")
print(f" - Low-Value Transaction Flagging Rate:  {low_val_alert_rate:.4f}")
print(f" - Disparate Impact Ratio:                {disparate_impact_ratio:.4f}")
print("\nDomain Note: A ratio between 0.80 and 1.25 indicates that your model treats financial size classes equitably.")